In [31]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Lambda
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split

In [2]:
base_path = "processed_signature/processed"
csv_path = os.path.join(base_path, "signature_dataset.csv")

df = pd.read_csv(csv_path)

print(df.head())

            image_path  person_id  label split
0  test/049/01_049.png         49      0  test
1  test/049/02_049.png         49      0  test
2  test/049/03_049.png         49      0  test
3  test/049/04_049.png         49      0  test
4  test/049/05_049.png         49      0  test


In [3]:
def load_image(img_path):
    img = image.load_img(os.path.join(base_path, img_path), target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    return img

In [4]:
print(df.columns)

Index(['image_path', 'person_id', 'label', 'split'], dtype='str')


In [5]:
#Create Pairs for Siamese Training

def create_pairs(df):
    pairs = []
    labels = []

    grouped = df.groupby("person_id")

    for writer_id, group in grouped:
        genuine = group[group['label'] == 0]['image_path'].values
        forged  = group[group['label'] == 1]['image_path'].values

        # Genuine-Genuine pairs (label=1)
        for i in range(len(genuine)-1):
            img1 = load_image(genuine[i])
            img2 = load_image(genuine[i+1])
            pairs.append([img1, img2])
            labels.append(1)

        # Genuine-Forged pairs (label=0)
        for i in range(min(len(genuine), len(forged))):
            img1 = load_image(genuine[i])
            img2 = load_image(forged[i])
            pairs.append([img1, img2])
            labels.append(0)

    return np.array(pairs), np.array(labels)

In [6]:
pairs, labels = create_pairs(df)

In [7]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    pairs, labels, test_size=0.2, random_state=42
)

# train_df = df[df["split"] == "train"]
# test_df  = df[df["split"] == "test"]

In [8]:
def l2_normalize(x):
    return K.l2_normalize(x, axis=1)

##  changes done here 

In [9]:
def build_embedding_model():

    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )

    # Freeze early layers
    for layer in base_model.layers[:-25]:
        layer.trainable = False

    x = base_model.output

    x = GlobalAveragePooling2D()(x)

    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    x = Dense(128, activation='relu')(x)

    x = Lambda(lambda x: K.l2_normalize(x, axis=1))(x)

    model = Model(base_model.input, x)

    return model

In [10]:
def euclidean_distance(vectors):
    x, y = vectors
    return K.sqrt(K.sum(K.square(x - y), axis=1, keepdims=True))

## changes done here 

In [11]:
embedding_model = build_embedding_model()

input_a = Input(shape=(224,224,3))
input_b = Input(shape=(224,224,3))

embedding_a = embedding_model(input_a)
embedding_b = embedding_model(input_b)

distance = Lambda(euclidean_distance)([embedding_a, embedding_b])

siamese_model = Model(inputs=[input_a, input_b], outputs=distance)

def contrastive_loss(y_true, y_pred):
    margin = 1
    return K.mean(
        y_true * K.square(y_pred) +
        (1 - y_true) * K.square(K.maximum(margin - y_pred, 0))
    )
optimizer = Adam(learning_rate=0.00005)

siamese_model.compile(
    loss="binary_crossentropy",
    optimizer=optimizer,
    metrics=['accuracy']
)
siamese_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 128)       │  2,619,840 │ input_layer_1[0]… │
│ (Functional)        │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 1)         │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,619,840 (9.99 MB)

 Trainable params: 1,723,264 (6.57 MB)

 Non-trainable params: 896,576 (3.42 MB)

In [12]:
#Balance Pairs
print("Positive pairs:", np.sum(labels==1))
print("Negative pairs:", np.sum(labels==0))


Positive pairs: 2085
Negative pairs: 1907


## changes done here

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

## changes done here

In [14]:
history = siamese_model.fit(

    [X_train[:,0], X_train[:,1]],
    y_train,

    validation_data=([X_test[:,0], X_test[:,1]], y_test),

    epochs=40,
    batch_size=16,

    callbacks=[early_stop, reduce_lr]
)

Epoch 1/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 172s 679ms/step - accuracy: 0.5208 - loss: 7.1979 - val_accuracy: 0.4994 - val_loss: 2.9803 - learning_rate: 5.0000e-05
Epoch 2/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 126s 628ms/step - accuracy: 0.5208 - loss: 6.6606 - val_accuracy: 0.5019 - val_loss: 3.7536 - learning_rate: 5.0000e-05
Epoch 3/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 119s 593ms/step - accuracy: 0.5208 - loss: 6.1816 - val_accuracy: 0.4919 - val_loss: 3.6948 - learning_rate: 5.0000e-05
Epoch 4/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 97s 486ms/step - accuracy: 0.5208 - loss: 5.4148 - val_accuracy: 0.4631 - val_loss: 3.0698 - learning_rate: 5.0000e-05
Epoch 5/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 94s 468ms/step - accuracy: 0.5208 - loss: 4.9866 - val_accuracy: 0.4618 - val_loss: 2.9288 - learning_rate: 1.5000e-05
Epoch 6/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 149s 501ms/step - accuracy: 0.5208 - loss: 4.7837 - val_accuracy: 0.4543 - val_loss: 2.8089 - learning_rate: 1.5000e-05
Epoch 7/40
200/200 ━━━━━━━━━━━━━━━━━━━━ 13

## changes done here

In [15]:
preds = siamese_model.predict([X_test[:,0], X_test[:,1]])

genuine = preds[y_test == 1]
forged  = preds[y_test == 0]

print("Mean Genuine Distance:", np.mean(genuine))
print("Mean Forged Distance:", np.mean(forged))

threshold = (np.mean(genuine) + np.mean(forged)) / 2

print("Optimal Threshold:", threshold)

25/25 ━━━━━━━━━━━━━━━━━━━━ 20s 739ms/step
Mean Genuine Distance: 0.5927162
Mean Forged Distance: 0.6712579
Optimal Threshold: 0.6319871


In [16]:
def get_embedding(img_path):
    img = load_image(img_path)
    img = np.expand_dims(img, axis=0)
    embedding = embedding_model.predict(img)
    return embedding

In [17]:
customer_database = {}

In [18]:
def register_customer(customer_id, signature_paths):
    embeddings = []
    for path in signature_paths:
        emb = get_embedding(path)
        embeddings.append(emb)
    customer_database[customer_id] = embeddings

In [19]:
def verify_signature(customer_id, test_signature_path, threshold):

    if customer_id not in customer_database:
        print("Customer not found!")
        return

    test_embedding = get_embedding(test_signature_path)

    stored_embeddings = customer_database[customer_id]

    distances = []

    for emb in stored_embeddings:
        dist = np.linalg.norm(test_embedding - emb)
        distances.append(dist)

    avg_distance = np.mean(distances)

    print("Average Distance:", avg_distance)

    if avg_distance < threshold:
        print("Signature Verified ✅")
    else:
        print("Forgery Detected ❌")

In [20]:
def get_embedding(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    embedding = embedding_model.predict(img)
    return embedding

In [21]:
import os

def register_customer(customer_id, folder_path):

    embeddings = []

    # Get all image files from folder
    image_files = [
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if file.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]

    if len(image_files) == 0:
        print("No images found in folder!")
        return

    for img_path in image_files:
        emb = get_embedding(img_path)
        embeddings.append(emb)

    customer_database[customer_id] = embeddings

    print(f"✅ Customer {customer_id} registered successfully.")
    print(f"Stored {len(embeddings)} signature samples.")

In [22]:
register_customer("C101", "processed")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 646ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
✅ Customer C101 registered successfully.
Stored 12 signature samples.


## changes done here

In [23]:
import numpy as np

def verify_signature(customer_id, test_signature_path, threshold=0.6):

    if customer_id not in customer_database:
        print("Customer not found!")
        return

    test_embedding = get_embedding(test_signature_path)

    stored_embeddings = customer_database[customer_id]

    distances = []

    for emb in stored_embeddings:
        dist = np.linalg.norm(test_embedding - emb)
        distances.append(dist)

    avg_distance = np.mean(distances)

    print("Average Distance:", avg_distance)

    if avg_distance < threshold:
        confidence = (1 - avg_distance/threshold) * 100
        print("✅ Genuine Signature")
        print("Confidence:", round(confidence,2), "%")
    else:
        confidence = (avg_distance/threshold - 1) * 100
        print("❌ Forged Signature")
        print("Confidence:", round(confidence,2), "%")

In [35]:
verify_signature(
    "C101",
    "forg.jpeg"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Average Distance: 0.610161
❌ Forged Signature
Confidence: 1.69 %


In [30]:
siamese_model.save("modified.h5")